# ORION Sparse Formula Ablation (Colab)

Dedicated, config-first notebook for tuning ORION sparse expander constants:
- `a = expander_head_linear_coeff`
- `b = expander_head_quadratic_coeff`
- `c = expander_s2_coeff`
- `d = expander_sh_coeff`

This notebook runs profile: `sparse_formula_ablation_t4`.

In [ ]:
# ==== User parameters ====
REPO_URL = "https://github.com/akashkguw/orion.git"
GIT_BRANCH = "main"

# Folder on Google Drive where the repo clone + runs are stored
DRIVE_PROJECT_DIR = "orion_formula_ablation"

PROFILE = "sparse_formula_ablation_t4"
OVERWRITE = False  # True: rerun trials even if run dirs already exist

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False


def run(cmd, cwd=None, check=True):
    print("+", " ".join(map(str, cmd)))
    return subprocess.run(cmd, cwd=cwd, check=check, text=True)


if IN_COLAB:
    drive.mount("/content/drive")
    workspace_root = Path("/content/drive/MyDrive") / DRIVE_PROJECT_DIR
else:
    workspace_root = Path.cwd()

workspace_root.mkdir(parents=True, exist_ok=True)
repo_root = workspace_root / "repo"
print("Workspace:", workspace_root)
print("Repo root:", repo_root)

In [ ]:
# Clone or update repository safely
if repo_root.exists():
    if (repo_root / ".git").exists():
        # Reuse existing clone
        run(["git", "fetch", "origin"], cwd=repo_root)
        run(["git", "checkout", GIT_BRANCH], cwd=repo_root)
        run(["git", "pull", "--ff-only", "origin", GIT_BRANCH], cwd=repo_root)
    else:
        raise RuntimeError(
            f"{repo_root} exists but is not a git repo. Rename or delete it, then rerun."
        )
else:
    run(["git", "clone", "--branch", GIT_BRANCH, REPO_URL, str(repo_root)])

os.chdir(repo_root)
print("Using:", Path.cwd())

In [ ]:
# Install project + analysis deps
run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"], cwd=repo_root)
run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        "requirements.txt",
        "-r",
        "requirements-dev.txt",
    ],
    cwd=repo_root,
)
run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=repo_root)
run([sys.executable, "-m", "pip", "install", "-q", "pandas", "matplotlib", "seaborn"], cwd=repo_root)

In [ ]:
# Environment check
import torch

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Run sparse formula ablation profile
cmd = [sys.executable, "-m", "orion.experiments", "--profile", PROFILE]
if OVERWRITE:
    cmd.append("--overwrite")

start = time.time()
run(cmd, cwd=repo_root)
print(f"Elapsed: {time.time() - start:.1f}s")

In [ ]:
# Locate latest run folder for this profile and load summary
import pandas as pd

runs_base = repo_root / "runs"
run_dirs = sorted(
    [p for p in runs_base.glob(f"{PROFILE}_dense_window_sparse_*") if p.is_dir()],
    key=lambda p: p.stat().st_mtime,
)
if not run_dirs:
    raise RuntimeError(f"No runs found under {runs_base} for profile {PROFILE}")

latest_run = run_dirs[-1]
summary_csv = latest_run / "summary.csv"
if not summary_csv.exists():
    raise RuntimeError(f"Missing summary CSV: {summary_csv}")

print("Latest run:", latest_run)
print("Summary CSV:", summary_csv)

df = pd.read_csv(summary_csv)
print("Rows:", len(df))
display(df.head(10))

In [ ]:
# Quick aggregate summary

from orion.experiments import paired_analysis_tables, prepare_analysis_columns, select_winners

df_ok = df[df["status"] == "ok"].copy()
df_ok = prepare_analysis_columns(df_ok)

print("Successful trials:", len(df_ok), "/", len(df))

group_cols = ["backend", "sparse_tag", "seq_len"]
summary = (
    df_ok.groupby(group_cols, as_index=False)
    .agg(
        runs=("trial_id", "count"),
        val_ppl_mean=("val_ppl", "mean"),
        val_ppl_std=("val_ppl", "std"),
        train_tok_s_mean=("train_tok_per_s", "mean"),
        train_tok_s_std=("train_tok_per_s", "std"),
        peak_vram_gb_mean=("peak_vram_gb", "mean"),
        peak_vram_gb_std=("peak_vram_gb", "std"),
    )
    .sort_values(["seq_len", "backend", "sparse_tag"])
)
display(summary)

paired_all, paired_focus, paired_agg = paired_analysis_tables(df_ok)
print("Paired rows (all non-dense variants):", len(paired_all))
print("Paired rows (focused):", len(paired_focus))
display(paired_agg)

winners = select_winners(paired_agg, val_ppl_tolerance=0.20)
print("Candidate winners:")
display(winners)

In [ ]:
# Optional: render all numeric metric plots to a folder in this run
from orion.experiments import plot_all_numeric_metrics

plots_dir = latest_run / "plots_sparse_formula_ablation"
result = plot_all_numeric_metrics(df_ok, out_dir=plots_dir)
print("Saved plots to:", plots_dir)
print("Numeric metrics plotted:", len(result["metrics"]))